In [ ]:
import shapely
from MobilityHubDataObjects import *
import pandas as pd
import geopandas as gpd
import folium
import numpy as np
from pyproj import Transformer
import datetime as dt

#map_area = gpd.read_file("./rawData/LA_City_Boundary/City_Boundary.shp").to_crs("EPSG:4326").loc[0, "geometry"]
#map_area = gpd.read_file("./rawData/LA_Times_Neighborhood_Boundaries.geojson").loc[65, "geometry"]
map_area = gpd.read_file("./rawData/waynecounty.geojson").to_crs(4326).loc[0,"geometry"]
map_area

In [ ]:
gtfs_instance = GTFSDataObject(
    "./gtfs/waynecounty",
    "https://transit.land/api/v2/rest/feeds.json",
    dt.timedelta(days=10),
    dt.time(hour=10), #TODO: need to handle tz
    dt.time(hour=15),
    min_headway=9000,
    api_key_path="./rawData/TRANSITLAND_KEY",
)
await gtfs_instance.load_data(map_area, 4326)

In [ ]:
from osmnx import settings
fta_instance = FTAFacilityInventoryDataObject(
    "./rawData/2022 Facility Inventory.xlsx",
    "./rawData/tl_2023_us_state/tl_2023_us_state.shp"
)
fta_instance.load_data(map_area, 4326)

In [ ]:
citybikes_instance = CityBikesDataObject("http://api.citybik.es/")
citybikes_instance.load_data(map_area, 4326)

In [ ]:
afdc_instance = AFDCApiDataObject(constants.AFDC_DATA_URL, "./cache/afdc_cache.geojson", "./rawData/AFDC_API_KEY")
afdc_instance.load_data(map_area, 4326)

In [6]:
osm_instance = OSMDataObject("./cache/osmnx_cache", {"amenity": ["bicycle_parking"]})
osm_instance.load_data(map_area, 4326)

In [7]:
ejscreen_instance = EJScreenDataObject("rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb")
ejscreen_instance.load_data(map_area, 4326)

In [ ]:
from MobilityHubDataObjects.utils import get_scores_for_all_objects


scores = get_scores_for_all_objects([gtfs_instance, citybikes_instance, osm_instance, afdc_instance], ["GTFS", "Citybikes", "OSM", "AFDC"])
scores.head()

In [ ]:
display_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
)
ejscreen_instance.get_folium_plot().add_to(display_map)
afdc_instance.get_folium_plot().add_to(display_map)
fta_instance.get_folium_plot().add_to(display_map)
osm_instance.get_folium_plot().add_to(display_map)
gtfs_instance.get_folium_plot().add_to(display_map)
citybikes_instance.get_folium_plot().add_to(display_map)
display_map

In [ ]:
from MobilityHubDataObjects.utils import basic_circle_marker


score_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
)
folium.GeoJson(scores, marker=basic_circle_marker("green"), popup=folium.GeoJsonPopup(fields=["score", "type"])).add_to(score_map)
score_map